# NASDAQ Level 3 data Exploratory Data Analysis (EDA)

Exploratory analysis of one full trading day of Nasdaq TotalView-ITCH 5.0 data:
`S061226-v50.txt.gz`, 12 June 2026 (17.9 GB gzipped, ~1.3 billion messages, 12,810 order books).

**What the data looks like.** ITCH is Level 3 (order-by-order) data. Each book message is one event on one
order, identified by its order reference number: add (`A`/`F`), execute (`E`/`C`), partial cancel (`X`),
delete (`D`), replace (`U`). Replaying these events rebuilds every order book of the day. Session and symbol
messages (`S`, `R`, `H`, `h`) mark market phases, map locate codes to symbols and flag halts. The file is
parsed with [itch-book](https://github.com/groovg/itch-book) into Parquet tables and analysed with polars.
For the tool, the setup and what each event means, see [README.md](README.md).

**Goal.** Select a subset of symbols that is small enough to replay and test, and that is useful for:
- **covering different types of load** (most important): deep queues at a few price levels, very high
  message rates, wide and sparse books, heavy cancel/replace flow, and a realistic baseline;
- **load testing** the order book data structures with real order flow;
- **correctness validation**: replaying the chosen symbols must reproduce the recorded executions and the
  end-of-day book.

In [27]:
import random

import polars as pl
from IPython.display import Markdown, display

pl.Config.set_tbl_cols(-1)  # show every column instead of eliding the middle ones

polars.config.Config

In [28]:
DATA_PATH = "data/parquet"

messages = pl.scan_parquet(f"{DATA_PATH}/messages.parquet")
symbols = pl.scan_parquet(f"{DATA_PATH}/symbols.parquet")
trades = pl.scan_parquet(f"{DATA_PATH}/trades.parquet")
bbo = pl.scan_parquet(f"{DATA_PATH}/bbo.parquet")
halts = pl.scan_parquet(f"{DATA_PATH}/halts.parquet")
system_events = pl.scan_parquet(f"{DATA_PATH}/system_events.parquet")

In [29]:
TABLES = {
    "messages": messages,
    "symbols": symbols,
    "trades": trades,
    "bbo": bbo,
    "halts": halts,
    "system_events": system_events,
}


def table_sample(lf: pl.LazyFrame, rows: int, blocks: int = 100, block_size: int = 1_000, seed: int = 42) -> pl.DataFrame:
    """Random sample without reading the whole table: `blocks` slices of `block_size` consecutive rows
    at random offsets. Tables with at most `blocks * block_size` rows are returned whole."""
    if rows <= blocks * block_size:
        return lf.collect()
    rng = random.Random(seed)
    offsets = sorted(rng.sample(range(rows - block_size), blocks))
    return pl.concat([lf.slice(offset, block_size) for offset in offsets]).collect()


def overview(name: str, n_show: int = 10) -> None:
    """Attributes (column -> dtype), head, random rows and describe() of one table."""
    lf = TABLES[name]
    schema = lf.collect_schema()
    rows = lf.select(pl.len()).collect().item()
    sample = table_sample(lf, rows)
    sampled = sample.height < rows

    display(Markdown(f"### `{name}`: {rows:,} rows × {len(schema)} columns"))

    display(Markdown("#### Attributes"))
    # Object dtype only so the dtype names render without string quotes; the table is display-only.
    attributes = pl.DataFrame(
        {column: [str(dtype)] for column, dtype in schema.items()},
        schema={column: pl.Object for column in schema},
    )
    with pl.Config(tbl_hide_column_data_types=True, tbl_hide_dataframe_shape=True, fmt_str_lengths=100):
        display(attributes)

    display(Markdown("#### Head"))
    display(lf.head(n_show).collect())

    display(Markdown("#### Sample" + (f" (from {sample.height:,} random rows)" if sampled else "")))
    shown = sample.sample(min(n_show, sample.height), seed=42)
    display(shown.sort("seq") if "seq" in schema else shown)

    display(Markdown("#### Statistics" + (f" (on {sample.height:,} random rows)" if sampled else "")))
    display(sample.describe())

## Tables overview

A quick look at each table: which columns it has, what its rows look like and roughly what the values are.
**This is a highlight of the data, not precise numbers.** Any number used for a decision (e.g. symbol
selection) must come from a dedicated query over the full table, like `events_per_symbol` below.

For each table:
- **title**: the exact number of rows and columns;
- **Attributes**: every column and its type;
- **Head**: the first 10 rows of the table;
- **Sample**: 10 random rows;
- **Statistics**: `describe()`, i.e. count, nulls, mean, std, min, quartiles and max per column.

`symbols`, `halts` and `system_events` are small and loaded whole, so their Sample and Statistics are exact.

`messages` (1.29B rows), `bbo` (491M) and `trades` (28M) are too big to load into memory. To keep the
overview fast (a few seconds, a few hundred MB), their Sample and Statistics use 100,000 rows: 100 blocks of
1,000 consecutive rows taken at random positions in the table. For these three tables:
- count and nulls describe the sample, not the table;
- min and max are those of the sample: the true extremes are almost certainly missed;
- mean, std and quartiles are rough estimates: rows in one block come from the same few milliseconds of the
  day, so they are less precise than 100,000 independent rows would be.

In [30]:
overview("messages")

### `messages`: 1,289,917,161 rows × 14 columns

#### Attributes

ts_event,seq,locate,symbol,type,action,side,price,size,remaining,printable,order_id,old_order_id,mpid
"Datetime(time_unit='ns', time_zone='UTC')",UInt64,UInt16,Categorical,String,String,String,Int64,UInt32,UInt32,Boolean,UInt64,UInt64,UInt16


#### Head

ts_event,seq,locate,symbol,type,action,side,price,size,remaining,printable,order_id,old_order_id,mpid
"datetime[ns, UTC]",u64,u16,cat,str,str,str,i64,u32,u32,bool,u64,u64,u16
2026-06-12 08:00:00.000425736 UTC,309047,9268,"""QID""","""A""","""A""","""B""",146000,100,100,true,3,0,0
2026-06-12 08:00:00.000427018 UTC,309048,10087,"""SDS""","""A""","""A""","""B""",589000,100,100,true,4,0,0
2026-06-12 08:00:00.000443178 UTC,309049,3039,"""DIA""","""A""","""A""","""B""",5103800,100,100,true,2,0,0
2026-06-12 08:00:00.000444173 UTC,309050,9315,"""QQQ""","""A""","""A""","""B""",7159500,100,100,true,7,0,0
2026-06-12 08:00:00.000445388 UTC,309051,10622,"""SPXS""","""A""","""A""","""B""",277700,100,100,true,8,0,0
2026-06-12 08:00:00.000458729 UTC,309052,9276,"""QLD""","""A""","""A""","""B""",920000,100,100,true,11,0,0
2026-06-12 08:00:00.000460246 UTC,309053,10624,"""SPXU""","""A""","""A""","""B""",389800,100,100,true,12,0,0
2026-06-12 08:00:00.000461469 UTC,309054,6066,"""IVV""","""A""","""A""","""B""",7417700,100,100,true,6,0,0
2026-06-12 08:00:00.000472652 UTC,309055,9315,"""QQQ""","""A""","""A""","""S""",7161400,100,100,true,15,0,0


#### Sample (from 100,000 random rows)

ts_event,seq,locate,symbol,type,action,side,price,size,remaining,printable,order_id,old_order_id,mpid
"datetime[ns, UTC]",u64,u16,cat,str,str,str,i64,u32,u32,bool,u64,u64,u16
2026-06-12 13:50:13.493311485 UTC,172349525,9315,"""QQQ""","""A""","""A""","""S""",7126300,20,20,true,221991983,0,0
2026-06-12 14:01:06.042339790 UTC,223520293,5722,"""IGV""","""D""","""C""","""S""",902800,25,0,true,357497438,0,0
2026-06-12 14:49:29.455101287 UTC,417392819,8095,"""NVDA""","""D""","""C""","""B""",2042800,4,0,true,541928175,0,0
2026-06-12 15:41:38.999511431 UTC,626946045,11388,"""TSL""","""A""","""A""","""S""",155800,200,200,true,1122120300,0,0
2026-06-12 15:54:32.754509699 UTC,681658897,6154,"""JBGS""","""A""","""A""","""B""",144800,2,2,true,1114412506,0,0
2026-06-12 16:22:37.032043835 UTC,801627181,3868,"""EXC""","""D""","""C""","""S""",461100,25,0,true,1308989850,0,0
2026-06-12 17:06:23.024003367 UTC,914880547,2762,"""CVX""","""E""","""F""","""B""",1870400,12,28,true,1497344758,0,0
2026-06-12 17:37:55.502803360 UTC,981429813,5115,"""GXO""","""A""","""A""","""B""",496800,90,90,true,1607668290,0,0
2026-06-12 19:36:43.289269794 UTC,1205387088,6677,"""LCTU""","""A""","""A""","""B""",792100,200,200,true,1558504195,0,0


#### Statistics (on 100,000 random rows)

statistic,ts_event,seq,locate,symbol,type,action,side,price,size,remaining,printable,order_id,old_order_id,mpid
str,str,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""","""100000""",100000.0,100000.0,"""100000""","""100000""","""100000""","""100000""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2026-06-12 15:47:38.543132+00:…",6.0230e8,7136.21151,null,null,null,null,1.4956e6,248.72758,158.66806,1.0,9.1864e8,2.7226e8,0.14492
"""std""",null,3.5404e8,3706.943835,null,null,null,null,2.0640e6,3301.277168,3239.75123,null,5.8024e8,5.2303e8,2.545428
"""min""","""2026-06-12 09:21:21.690899+00:…",1.4333817e7,1.0,null,"""A""","""A""","""B""",1.0,1.0,0.0,1.0,75705.0,0.0,0.0
"""25%""","""2026-06-12 14:22:11.631716+00:…",3.10726889e8,4080.0,null,null,null,null,326300.0,40.0,0.0,null,4.41565676e8,0.0,0.0
"""50%""","""2026-06-12 15:27:24.782998+00:…",5.74028752e8,7530.0,null,null,null,null,776100.0,100.0,40.0,null,8.50116292e8,0.0,0.0
"""75%""","""2026-06-12 16:30:12.494841+00:…",8.23096278e8,10517.0,null,null,null,null,1.6177e6,100.0,100.0,null,1.3005e9,2.84846303e8,0.0
"""max""","""2026-06-12 19:58:32.272777+00:…",1.2815e9,12809.0,null,"""X""","""M""","""S""",5.2401e7,9.99999e5,9.99999e5,1.0,2.3504e9,2.3504e9,90.0


In [31]:
overview("symbols")

### `symbols`: 12,809 rows × 15 columns

#### Attributes

locate,symbol,market_category,financial_status,round_lot_size,round_lots_only,issue_classification,issue_subtype,authenticity,short_sale_threshold,ipo_flag,luld_tier,etp_flag,etp_leverage,inverse
UInt16,String,String,String,UInt32,Boolean,String,String,String,String,String,String,String,UInt32,String


#### Head

locate,symbol,market_category,financial_status,round_lot_size,round_lots_only,issue_classification,issue_subtype,authenticity,short_sale_threshold,ipo_flag,luld_tier,etp_flag,etp_leverage,inverse
u16,str,str,str,u32,bool,str,str,str,str,str,str,str,u32,str
1,"""A""","""N""",""" """,100,false,"""C""","""Z""","""P""","""N""",""" ""","""1""","""N""",0,"""N"""
2,"""AA""","""N""",""" """,100,false,"""C""","""Z""","""P""","""N""",""" ""","""1""","""N""",0,"""N"""
3,"""AAA""","""P""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""
4,"""AAAA""","""Z""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""
5,"""AAAC""","""P""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""
6,"""AAAD""","""P""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""
7,"""AAAU""","""Z""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""1""","""Y""",0,"""N"""
8,"""AACB""","""G""","""D""",100,false,"""O""","""Z""","""P""","""N""","""N""","""2""","""N""",0,"""N"""
9,"""AACBR""","""G""","""N""",100,false,"""R""","""Z""","""P""","""N""","""N""",""" ""","""N""",0,"""N"""


#### Sample

locate,symbol,market_category,financial_status,round_lot_size,round_lots_only,issue_classification,issue_subtype,authenticity,short_sale_threshold,ipo_flag,luld_tier,etp_flag,etp_leverage,inverse
u16,str,str,str,u32,bool,str,str,str,str,str,str,str,u32,str
10424,"""SMZ""","""Z""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""
4082,"""FETH""","""Z""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""1""","""Y""",0,"""N"""
12596,"""XT""","""G""","""N""",100,false,"""Q""","""I""","""P""","""N""","""N""","""1""","""Y""",1,"""N"""
8977,"""PQJL""","""G""","""N""",100,false,"""Q""","""I""","""P""","""N""","""N""","""2""","""Y""",1,"""N"""
10161,"""SFLR""","""P""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""1""","""Y""",0,"""N"""
7531,"""MSW""","""S""","""N""",100,false,"""O""","""Z""","""P""","""N""","""N""","""2""","""N""",0,"""N"""
1606,"""BPACU""","""S""","""N""",100,false,"""U""","""Z""","""P""","""N""","""N""","""2""","""N""",0,"""N"""
7750,"""NCRA""","""S""","""D""",100,false,"""C""","""Z""","""P""","""N""","""N""","""2""","""N""",0,"""N"""
2661,"""CSRE""","""P""",""" """,100,false,"""Q""","""I""","""P""","""N""",""" ""","""2""","""Y""",0,"""N"""


#### Statistics

statistic,locate,symbol,market_category,financial_status,round_lot_size,round_lots_only,issue_classification,issue_subtype,authenticity,short_sale_threshold,ipo_flag,luld_tier,etp_flag,etp_leverage,inverse
str,f64,str,str,str,f64,f64,str,str,str,str,str,str,str,f64,str
"""count""",12809.0,"""12809""","""12809""","""12809""",12809.0,12809.0,"""12809""","""12809""","""12809""","""12809""","""12809""","""12809""","""12809""",12809.0,"""12809"""
"""null_count""",0.0,"""0""","""0""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0"""
"""mean""",6405.0,null,null,null,98.716527,0.0,null,null,null,null,null,null,null,0.120228,null
"""std""",3697.784134,null,null,null,8.920933,null,null,null,null,null,null,null,null,0.379319,null
"""min""",1.0,"""A""","""A""",""" """,10.0,0.0,"""A""","""A""","""P""","""N""",""" """,""" ""","""N""",0.0,"""N"""
"""25%""",3203.0,null,null,null,100.0,null,null,null,null,null,null,null,null,0.0,null
"""50%""",6405.0,null,null,null,100.0,null,null,null,null,null,null,null,null,0.0,null
"""75%""",9607.0,null,null,null,100.0,null,null,null,null,null,null,null,null,0.0,null
"""max""",12809.0,"""ZYME""","""Z""","""N""",100.0,0.0,"""W""","""Z""","""T""","""Y""","""Y""","""2""","""Y""",3.0,"""Y"""


In [32]:
overview("trades")

### `trades`: 28,269,323 rows × 12 columns

#### Attributes

ts_event,seq,locate,symbol,kind,price,size,side,order_id,match_number,cross_type,broken
"Datetime(time_unit='ns', time_zone='UTC')",UInt64,UInt16,Categorical,String,Int64,UInt64,String,UInt64,UInt64,String,Boolean


#### Head

ts_event,seq,locate,symbol,kind,price,size,side,order_id,match_number,cross_type,broken
"datetime[ns, UTC]",u64,u16,cat,str,i64,u64,str,u64,u64,str,bool
2026-06-12 08:00:00.012083922 UTC,309652,5875,"""INTC""","""E""",1192900,3500,"""B""",2202,25597,"""N""",false
2026-06-12 08:00:00.020744411 UTC,309934,8117,"""NVO""","""E""",440600,10,"""B""",147,25598,"""N""",false
2026-06-12 08:00:00.020773473 UTC,309935,8117,"""NVO""","""E""",440600,10,"""B""",147,25599,"""N""",false
2026-06-12 08:00:00.026793266 UTC,309993,4910,"""GOOGL""","""E""",3591300,4,"""S""",2074,25600,"""N""",false
2026-06-12 08:00:00.026799314 UTC,310000,4910,"""GOOGL""","""E""",3591300,1,"""S""",2074,25601,"""N""",false
2026-06-12 08:00:00.026938658 UTC,310143,5388,"""HSHP""","""E""",148100,5,"""B""",2146,25602,"""N""",false
2026-06-12 08:00:00.026968384 UTC,310153,5316,"""HMY""","""E""",150600,1650,"""B""",3306,25603,"""N""",false
2026-06-12 08:00:00.027281321 UTC,310169,807,"""ASML""","""P""",18552300,1,"""N""",0,25604,"""N""",false
2026-06-12 08:00:00.036696910 UTC,310472,5316,"""HMY""","""E""",151000,57,"""B""",4086,25605,"""N""",false


#### Sample (from 100,000 random rows)

ts_event,seq,locate,symbol,kind,price,size,side,order_id,match_number,cross_type,broken
"datetime[ns, UTC]",u64,u16,cat,str,i64,u64,str,u64,u64,str,bool
2026-06-12 13:49:18.144545503 UTC,167048904,2709,"""CTSH""","""E""",509900,20,"""B""",264966906,3780163,"""N""",false
2026-06-12 14:07:43.029559634 UTC,253678705,4398,"""FRMI""","""P""",69800,26,"""N""",0,5351612,"""N""",false
2026-06-12 14:51:58.965733121 UTC,427323684,10517,"""SOXX""","""E""",5890000,10,"""B""",770966588,8319136,"""N""",false
2026-06-12 16:30:00.541145469 UTC,822417515,4039,"""FE""","""E""",466300,52,"""S""",1343504150,16634407,"""N""",false
2026-06-12 17:06:05.773950320 UTC,910914485,7564,"""MTUM""","""P""",3255200,100,"""N""",0,18380487,"""N""",false
2026-06-12 18:21:28.260921174 UTC,1066123356,938,"""AVGO""","""P""",3799700,40,"""N""",0,21682693,"""N""",false
2026-06-12 19:06:35.842282346 UTC,1145177286,178,"""ADBE""","""E""",2025000,51,"""S""",1154173097,23614416,"""N""",false
2026-06-12 19:16:47.710664656 UTC,1163621391,7797,"""NESR""","""E""",262300,3,"""B""",1499990379,24132682,"""N""",false
2026-06-12 19:54:00.052424514 UTC,1257108527,1058,"""BAK""","""P""",36750,1,"""N""",0,27045093,"""N""",false


#### Statistics (on 100,000 random rows)

statistic,ts_event,seq,locate,symbol,kind,price,size,side,order_id,match_number,cross_type,broken
str,str,f64,f64,str,str,f64,f64,str,f64,f64,str,f64
"""count""","""100000""",100000.0,100000.0,"""100000""","""100000""",100000.0,100000.0,"""100000""",100000.0,100000.0,"""100000""",100000.0
"""null_count""","""0""",0.0,0.0,"""0""","""0""",0.0,0.0,"""0""",0.0,0.0,"""0""",0.0
"""mean""","""2026-06-12 16:26:41.525247+00:…",7.0266e8,6823.65472,null,null,1.5097e6,84.21576,null,7.3261e8,1.4537e7,null,0.0
"""std""",null,4.0927e8,3850.974364,null,null,2.3038e6,448.879913,null,7.2823e8,8.6699e6,null,null
"""min""","""2026-06-12 09:44:03.095830+00:…",1.7863746e7,1.0,null,"""C""",115.0,1.0,"""B""",0.0,243659.0,"""N""",0.0
"""25%""","""2026-06-12 14:36:45.424777+00:…",3.66641691e8,3313.0,null,null,255050.0,7.0,null,0.0,7.341788e6,null,null
"""50%""","""2026-06-12 15:55:24.148010+00:…",6.84628317e8,7466.0,null,null,798500.0,28.0,null,5.41110227e8,1.38398e7,null,null
"""75%""","""2026-06-12 18:47:16.916191+00:…",1.1100e9,10536.0,null,null,1.6861e6,100.0,null,1.2775e9,2.2732512e7,null,null
"""max""","""2026-06-12 19:59:59.120437+00:…",1.2925e9,12809.0,null,"""P""",5.23288e7,50000.0,"""S""",2.3677e9,2.9415326e7,"""N""",0.0


In [33]:
overview("bbo")

### `bbo`: 491,197,723 rows × 10 columns

#### Attributes

ts_event,seq,locate,symbol,bid_px,bid_sz,bid_ct,ask_px,ask_sz,ask_ct
"Datetime(time_unit='ns', time_zone='UTC')",UInt64,UInt16,Categorical,Int64,UInt32,UInt32,Int64,UInt32,UInt32


#### Head

ts_event,seq,locate,symbol,bid_px,bid_sz,bid_ct,ask_px,ask_sz,ask_ct
"datetime[ns, UTC]",u64,u16,cat,i64,u32,u32,i64,u32,u32
2026-06-12 08:00:00.000425736 UTC,309047,9268,"""QID""",146000,100,1,0,0,0
2026-06-12 08:00:00.000427018 UTC,309048,10087,"""SDS""",589000,100,1,0,0,0
2026-06-12 08:00:00.000443178 UTC,309049,3039,"""DIA""",5103800,100,1,0,0,0
2026-06-12 08:00:00.000444173 UTC,309050,9315,"""QQQ""",7159500,100,1,0,0,0
2026-06-12 08:00:00.000445388 UTC,309051,10622,"""SPXS""",277700,100,1,0,0,0
2026-06-12 08:00:00.000458729 UTC,309052,9276,"""QLD""",920000,100,1,0,0,0
2026-06-12 08:00:00.000460246 UTC,309053,10624,"""SPXU""",389800,100,1,0,0,0
2026-06-12 08:00:00.000461469 UTC,309054,6066,"""IVV""",7417700,100,1,0,0,0
2026-06-12 08:00:00.000472652 UTC,309055,9315,"""QQQ""",7159500,100,1,7161400,100,1


#### Sample (from 100,000 random rows)

ts_event,seq,locate,symbol,bid_px,bid_sz,bid_ct,ask_px,ask_sz,ask_ct
"datetime[ns, UTC]",u64,u16,cat,i64,u32,u32,i64,u32,u32
2026-06-12 13:42:31.472316991 UTC,129048957,5039,"""GSPY""",400500,300,1,401400,400,2
2026-06-12 14:00:17.337977755 UTC,219185991,9304,"""QPUX""",313300,200,1,316600,200,1
2026-06-12 14:26:37.949190142 UTC,328706628,12092,"""VTV""",2164600,140,4,2164800,50,3
2026-06-12 16:02:49.742380603 UTC,716866806,11947,"""VIXY""",242500,45,3,242600,176,5
2026-06-12 16:15:13.813438470 UTC,773152089,7232,"""MGK""",867800,286,3,868000,339,3
2026-06-12 16:57:40.093895731 UTC,892556320,11228,"""TNA""",702700,60,2,702900,120,3
2026-06-12 17:47:31.417295779 UTC,999954132,758,"""ARR""",171400,1004,9,171600,600,5
2026-06-12 18:06:22.006794096 UTC,1035213333,11021,"""TEAM""",880100,37,2,880800,100,1
2026-06-12 19:34:50.199185607 UTC,1200019000,3225,"""DT""",407000,225,3,407200,146,4


#### Statistics (on 100,000 random rows)

statistic,ts_event,seq,locate,symbol,bid_px,bid_sz,bid_ct,ask_px,ask_sz,ask_ct
str,str,f64,f64,str,f64,f64,f64,f64,f64,f64
"""count""","""100000""",100000.0,100000.0,"""100000""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2026-06-12 15:46:37.016962+00:…",6.0172e8,7200.96858,null,1.1880e6,2069.16463,3.60933,1.1897e6,2056.20619,3.5749
"""std""",null,3.8129e8,3731.076272,null,1.8547e6,12217.309269,4.229582,1.8560e6,11800.014462,4.055253
"""min""","""2026-06-12 08:37:42.412222+00:…",8.143122e6,1.0,null,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""","""2026-06-12 14:18:34.536613+00:…",2.96318656e8,4013.0,null,261700.0,100.0,1.0,262900.0,100.0,1.0
"""50%""","""2026-06-12 15:13:44.471837+00:…",5.34793189e8,7528.0,null,587100.0,210.0,2.0,587600.0,216.0,2.0
"""75%""","""2026-06-12 17:15:57.310720+00:…",9.38765744e8,10622.0,null,1.2149e6,691.0,5.0,1.2153e6,700.0,5.0
"""max""","""2026-06-12 19:55:28.905870+00:…",1.2650e9,12809.0,null,2.10904e7,465934.0,269.0,2.11065e7,351320.0,155.0


In [34]:
overview("halts")

### `halts`: 12,979 rows × 8 columns

#### Attributes

ts_event,seq,locate,symbol,kind,state,reason,market
"Datetime(time_unit='ns', time_zone='UTC')",UInt64,UInt16,Categorical,String,String,String,String


#### Head

ts_event,seq,locate,symbol,kind,state,reason,market
"datetime[ns, UTC]",u64,u16,cat,str,str,str,str
2026-06-12 07:04:57.196486632 UTC,65,1,"""A""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.198572799 UTC,132,2,"""AA""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.198983087 UTC,160,3,"""AAA""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.199528942 UTC,199,4,"""AAAA""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.202078561 UTC,202,5,"""AAAC""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.202154532 UTC,208,6,"""AAAD""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.202282816 UTC,219,7,"""AAAU""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.202444789 UTC,232,8,"""AACB""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.202642790 UTC,248,9,"""AACBR""","""H""","""T""","""""","""N"""


#### Sample

ts_event,seq,locate,symbol,kind,state,reason,market
"datetime[ns, UTC]",u64,u16,cat,str,str,str,str
2026-06-12 07:04:57.333030164 UTC,14856,1627,"""BRC""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.360845829 UTC,18201,2702,"""CTOR""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.387993896 UTC,21081,4142,"""FHTX""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.477575195 UTC,28071,7645,"""MYMF""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.480916122 UTC,28515,7867,"""NIHI""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.506647079 UTC,31001,9113,"""PSQO""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.532845518 UTC,33399,10313,"""SKYY""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.536661277 UTC,33933,10581,"""SPMO""","""H""","""T""","""""","""N"""
2026-06-12 07:04:57.563543550 UTC,37037,12133,"""WAGN""","""H""","""T""","""""","""N"""


#### Statistics

statistic,ts_event,seq,locate,symbol,kind,state,reason,market
str,str,f64,f64,str,str,str,str,str
"""count""","""12979""",12979.0,12979.0,"""12979""","""12979""","""12979""","""12979""","""12979"""
"""null_count""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""","""0"""
"""mean""","""2026-06-12 07:12:21.437877+00:…",8.1575e6,6394.724093,null,null,null,null,null
"""std""",null,8.3005e7,3705.799154,null,null,null,null,null
"""min""","""2026-06-12 07:04:57.196486+00:…",65.0,1.0,null,"""H""","""H""","""""","""N"""
"""25%""","""2026-06-12 07:04:57.371109+00:…",19301.0,3177.0,null,null,null,null,null
"""50%""","""2026-06-12 07:04:57.459482+00:…",25789.0,6376.0,null,null,null,null,null
"""75%""","""2026-06-12 07:04:57.517080+00:…",32279.0,9610.0,null,null,null,null,null
"""max""","""2026-06-12 23:50:00.000308+00:…",1.3042e9,12809.0,null,"""H""","""T""","""T3""","""N"""


In [35]:
overview("system_events")

### `system_events`: 6 rows × 3 columns

#### Attributes

ts_event,seq,event
"Datetime(time_unit='ns', time_zone='UTC')",UInt64,String


#### Head

ts_event,seq,event
"datetime[ns, UTC]",u64,str
2026-06-12 07:01:32.008976642 UTC,1,"""O"""
2026-06-12 08:00:00.000105640 UTC,309046,"""S"""
2026-06-12 13:30:00.000052397 UTC,51563013,"""Q"""
2026-06-12 20:00:00.000026381 UTC,1292789520,"""M"""
2026-06-13 00:00:00.000018415 UTC,1304462501,"""E"""
2026-06-13 00:05:00.000020522 UTC,1304894064,"""C"""


#### Sample

ts_event,seq,event
"datetime[ns, UTC]",u64,str
2026-06-12 07:01:32.008976642 UTC,1,"""O"""
2026-06-12 08:00:00.000105640 UTC,309046,"""S"""
2026-06-12 13:30:00.000052397 UTC,51563013,"""Q"""
2026-06-12 20:00:00.000026381 UTC,1292789520,"""M"""
2026-06-13 00:00:00.000018415 UTC,1304462501,"""E"""
2026-06-13 00:05:00.000020522 UTC,1304894064,"""C"""


#### Statistics

statistic,ts_event,seq,event
str,str,f64,str
"""count""","""6""",6.0,"""6"""
"""null_count""","""0""",0.0,"""0"""
"""mean""","""2026-06-12 16:06:05.334866+00:…",6.5900e8,null
"""std""",null,7.0322e8,null
"""min""","""2026-06-12 07:01:32.008976+00:…",1.0,"""C"""
"""25%""","""2026-06-12 08:00:00.000105+00:…",309046.0,null
"""50%""","""2026-06-12 20:00:00.000026+00:…",1.2928e9,null
"""75%""","""2026-06-13 00:00:00.000018+00:…",1.3045e9,null
"""max""","""2026-06-13 00:05:00.000020+00:…",1.3049e9,"""S"""


In [36]:
events_per_symbol = (
    messages
    .group_by("locate")
    .agg(pl.len().alias("events"))
    .join(symbols.select("locate", "symbol"), on="locate", how="left")
    .select("symbol", "locate", "events")
    .sort("events", descending=True)
    .collect(engine="streaming")
)
events_per_symbol

symbol,locate,events
str,u16,u32
"""QQQ""",9315,28415132
"""SPY""",10627,19654959
"""SPCX""",10536,18813350
"""XLK""",12493,14156219
"""TQQQ""",11300,12780957
…,…,…
"""ZXIET""",12805,4
"""ZIEXT""",12747,4
"""IGZ""",5723,2
